# CAMSAP 2025: A Multiscale Geometric Method for Capturing Relational Topic Alignment

**Paper:** Conrad D. Hougen, Karl T. Pazdernik, Alfred O. Hero  
"A Multiscale Geometric Method for Capturing Relational Topic Alignment"  
*IEEE CAMSAP 2025*

---

This notebook reproduces the core experimental results from the CAMSAP2025 paper using the
MSTML library.  The pipeline can be split into two conceptual halves:

**GDLTM (Geometry-Driven Longitudinal Topic Model)** — *temporal topic analysis*
1. Data loading & filtering  
2. Text preprocessing & vocabulary optimisation  
3. Author disambiguation  
4. Temporal chunking  
5. Per-chunk LDA ensemble  

**MSTML (Multiscale Topic Manifold Learning)** — *relational alignment*
6. Author/document topic distributions (barycenters)  
7. Diffusion over the topic k-NN graph  
8. Topic manifold (Hellinger distance + Ward hierarchical clustering)  
9. PHATE embedding of the topic manifold  
10. Meta-topic clustering & visualisation  

---

### Experimental Parameters (CAMSAP2025)

| Parameter | Value |
|---|---|
| Dataset | arXiv |
| Date range | 2012-01-01 – 2023-12-31 |
| Categories | `cs.LG`, `stat.AP`, `stat.CO`, `stat.ME`, `stat.ML`, `stat.OT`, `stat.TH` |
| Diffusion k-NN (`ct_distn_diffusion_knnk`) | 5 |
| Dendrogram cut height | 0.68 |
| Topic manifold k-NN (`knnk_for_tpc_dendro`) | 100 |
| Temporal chunk size | 1 month (see config) |
| Num topics | auto (docs_per_topic=100) |

### Prerequisites

- arXiv JSON snapshot downloaded to `data/arxiv/original/`  
  (download from [Kaggle arXiv Dataset](https://www.kaggle.com/datasets/Cornell-University/arxiv))
- All dependencies installed: `python build.py`
- Adjust `experiment_directory` below to a path with sufficient disk space

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from mstml.core import MstmlOrchestrator

## 1. Configuration

Set the paths and CAMSAP2025 parameters here.  All other hyperparameters come
from `mstml/config.yaml`, which you can edit to override defaults.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATASET_NAME      = "arxiv"
EXPERIMENT_NAME   = "camsap2025"
EXPERIMENT_DIR    = os.path.join("..", "experiments", "camsap2025_rta")

# ── arXiv schema mapping ──────────────────────────────────────────────────────
# Maps arXiv JSONL field names to MSTML standard field names.
ARXIV_SCHEMA_MAP = {
    "abstract":       "raw_text",
    "update_date":    "date",
    "authors_parsed": "authors",
}

# ── CAMSAP2025 corpus parameters ──────────────────────────────────────────────
DATE_RANGE  = {"start": "2012-01-01", "end": "2023-12-31"}
CATEGORIES  = ["cs.LG", "stat.AP", "stat.CO", "stat.ME", "stat.ML", "stat.OT", "stat.TH"]

# ── CAMSAP2025 key hyperparameters (override config.yaml defaults if needed) ──
MONTHS_PER_CHUNK       = 1     # Monthly chunks for temporal resolution
DIFFUSION_KNN          = 5     # ct_distn_diffusion_knnk
MANIFOLD_CUT_HEIGHT    = 0.68  # Dendrogram cut for meta-topic clusters

print(f"Experiment directory: {os.path.abspath(EXPERIMENT_DIR)}")

## 2. Initialise Orchestrator

The `MstmlOrchestrator` is the single entry point for the pipeline.  Hyperparameters
that are not passed explicitly default to `mstml/config.yaml`.

In [ ]:
orch = MstmlOrchestrator(
    dataset_name=DATASET_NAME,
    experiment_name=EXPERIMENT_NAME,
    experiment_directory=EXPERIMENT_DIR,
)

# Override diffusion knn in config to match CAMSAP2025
orch.config["author_doc_embeddings"]["ct_distn_diffusion_knnk"] = DIFFUSION_KNN

# Configure corpus filters
orch.configure_data_filters(
    date_range=DATE_RANGE,
    categories=CATEGORIES,
)

## 3. Data Loading

Loads and validates the arXiv JSONL corpus.  The `overwrite=False` flag means
a previously saved `main_df.pkl` is reused if it exists — safe to re-run.

The schema map tells the loader how to rename arXiv's field names (`abstract`,
`update_date`, `authors_parsed`) to the MSTML standard (`raw_text`, `date`,
`authors`).

In [ ]:
orch.load_raw_data(input_schema_map=ARXIV_SCHEMA_MAP, overwrite=False)
print(f"Total documents loaded: {len(orch.documents_df):,}")

In [ ]:
orch.apply_data_filters()
print(f"Documents after filtering: {len(orch.documents_df):,}")
orch.documents_df[["title", "date", "categories"]].tail(5)

## 4. Text Preprocessing

Multi-step pipeline from GDLTM:

1. **Tokenisation & lemmatisation** — Gensim + NLTK WordNetLemmatizer  
2. **Frequency filtering** — remove tokens appearing in ≤ `low_thresh` docs
   or > `high_frac` of docs  
3. **LDA-based term relevancy** — train a global LDA model, keep the top-N
   terms by λ-weighted relevancy score  
   (λ=0.6, top_terms=2000, as in `config.yaml`)

This reproduces the vocabulary construction described in §III-A of the paper.

In [ ]:
orch.preprocess_text()
print("Preprocessing complete")

## 5. Author Disambiguation

TF-IDF cosine similarity over character 3-grams with a 0.90 threshold,
as described in §III-B.  Assigns unique 7-digit IDs to author clusters.

In [ ]:
orch.apply_author_disambiguation()
n_unique_authors = orch.documents_df["author_ids"].explode().nunique()
print(f"Unique disambiguated authors: {n_unique_authors:,}")

## 6. Co-author Network

Constructs a temporal co-author network from shared document authorship.
The `temporal=True` flag creates per-chunk snapshots that are used for
the diffusion step in §III-D.

In [ ]:
orch.setup_coauthor_network(temporal=True, overwrite=False)
import networkx as nx
G = orch.coauthor_network
print(f"Co-author network: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")

## 7. Temporal Chunking

Partition documents into 1-month windows (CAMSAP2025 setting).
The temporal smoothing decay γ=0.75 controls how topic similarity
degrades across time — see §III-C.

In [ ]:
orch.create_temporal_chunks(
    months_per_chunk=MONTHS_PER_CHUNK,
    temporal_smoothing_decay=0.75,
)
print(f"Number of temporal chunks: {len(orch.time_chunks)}")
chunk_sizes = [len(c) for c in orch.time_chunks]
print(f"Chunk sizes — min: {min(chunk_sizes)}, max: {max(chunk_sizes)}, mean: {sum(chunk_sizes)/len(chunk_sizes):.0f}")

## 8. Temporal LDA Ensemble

Train a separate LDA model on each monthly chunk.  The number of topics per
chunk is set automatically (`num_topics=auto`) from `docs_per_topic=100`.

This produces the per-chunk topic-word matrices Φ₁, Φ₂, …, Φ_T used
throughout the rest of the MSTML pipeline.

In [ ]:
orch.train_chunk_models(overwrite=False)
total_topics = len(orch.topic_vectors)
print(f"Total topics across all chunks: {total_topics:,}")
print(f"Topic vector shape (Φ matrix): {orch.topic_vectors.shape}")

## 9. Author/Document Topic Distributions

Each document's LDA distribution θ_d is expanded into the full
joint topic space across all chunks (zero-padded outside its chunk).
Author distributions are computed as weighted barycenters:

$$\psi_u^{(0)} = \sum_{d \ni u} w_{u,d} \cdot \theta_d$$

where $w_{u,d} = 1 / |\text{authors}(d)|$ (inverse co-author count weighting),
as described in §III-C.

In [ ]:
orch.build_author_document_distributions()
print(f"Author distributions computed: {len(orch.author_topic_barycenters):,} authors")
print(f"Document distributions computed: {len(orch.expanded_doc_topic_distns):,} docs")

## 10. Diffusion Over the Topic Graph

Smooth author/document distributions by propagating probability mass
through the k-NN topic graph (k=5, CAMSAP2025 setting).

The diffusion kernel is built using FAISS on √Φ (Hellinger approximation).
One iteration of message-passing at rate γ=0.7:

$$\psi_u^{(1)}[j] = \begin{cases} \psi_u^{(0)}[j] & \text{if } \psi_u^{(0)}[j] > 0 \\
\gamma \sum_{k \in \mathcal{N}(j)} w_{jk} \cdot \psi_u^{(0)}[k] & \text{otherwise}
\end{cases}$$

This corresponds to §III-D in the paper.

In [ ]:
orch.apply_diffusion()
print(f"Diffused author distributions: {len(orch.author_ct_distns):,}")
print(f"Diffusion matrix shape: {orch.diffusion_matrix.shape}, nnz: {orch.diffusion_matrix.nnz:,}")

## 11. Topic Manifold — Hellinger Distance + Ward Clustering

Construct the topic manifold by:
1. Computing pairwise Hellinger distances between all Φ_t vectors via FAISS
   (k=100 nearest neighbours, exact Hellinger = √(FAISS_L2/2))
2. Ward hierarchical clustering to produce the topic dendrogram

The dendrogram encodes multiscale topic structure used for meta-topic extraction
at cut height h=0.68 (§III-E).

In [ ]:
orch.build_topic_manifold()
print(f"Linkage matrix shape: {orch.topic_dendrogram_linkage.shape}")
print(f"Min cut height: {orch.min_cut_height:.4f}")
print(f"Max cut height: {orch.max_cut_height:.4f}")
print(f"CAMSAP2025 cut height: {MANIFOLD_CUT_HEIGHT} "
      f"({'within range' if orch.min_cut_height < MANIFOLD_CUT_HEIGHT < orch.max_cut_height else 'OUT OF RANGE — adjust cut height'})")

### Visualise the Dendrogram

The dendrogram x-axis is topic index; y-axis is Hellinger distance (∈ [0, 1]).
The red dashed line shows the CAMSAP2025 cut height h=0.68.

In [ ]:
from scipy.cluster.hierarchy import dendrogram

fig, ax = plt.subplots(figsize=(14, 4))
dendrogram(
    orch.topic_dendrogram_linkage,
    ax=ax,
    no_labels=True,
    color_threshold=MANIFOLD_CUT_HEIGHT,
)
ax.axhline(MANIFOLD_CUT_HEIGHT, color="red", linestyle="--",
           label=f"Cut height = {MANIFOLD_CUT_HEIGHT}")
ax.set_xlabel("Topic index")
ax.set_ylabel("Hellinger distance")
ax.set_title("Topic Dendrogram (Ward linkage, Hellinger distance)")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(EXPERIMENT_DIR, "fig_topic_dendrogram.pdf"), dpi=300)
plt.show()

## 12. PHATE Embedding of the Topic Manifold

Embed all Φ_t vectors into 2-D using PHATE (Potential of Heat diffusion for
Affinity-based Transition Embedding), coloured by meta-topic cluster.

The cut height h=0.68 partitions the dendrogram into meta-topic clusters,
each representing a coherent research theme.  This produces **Figure 2** from
the CAMSAP2025 paper.

In [ ]:
orch.create_topic_embedding(
    method="phate",
    n_components=2,
    cut_height=MANIFOLD_CUT_HEIGHT,
)
n_meta = len(set(orch.topic_cluster_labels.values())) if hasattr(orch, "topic_cluster_labels") else "?"
print(f"PHATE embedding shape: {orch.topic_embedding.shape}")
print(f"Meta-topic clusters at cut h={MANIFOLD_CUT_HEIGHT}: {n_meta}")

In [ ]:
# Figure 2: Topic space embedding coloured by meta-topic
orch.display_topic_embedding(
    color_by="meta_topic",
    title=f"MSTML Topic Space Embedding — arXiv cs.LG+stat 2012–2023\n"
          f"(PHATE, Ward h={MANIFOLD_CUT_HEIGHT})",
    save_path=os.path.join(EXPERIMENT_DIR, "fig_topic_embedding.pdf"),
)

## 13. Inspect Meta-Topic Clusters

Each meta-topic cluster groups semantically related topics from across all
time chunks.  Below we display the top-10 terms for each cluster's centroid.

In [ ]:
from collections import defaultdict
from mstml._math_driver import mstml_term_relevance_stable

if hasattr(orch, "topic_cluster_labels") and orch.topic_cluster_labels is not None:
    # Group topic indices by meta-topic cluster
    cluster_to_topics = defaultdict(list)
    for topic_idx, cluster_id in orch.topic_cluster_labels.items():
        cluster_to_topics[cluster_id].append(topic_idx)

    # Load vocabulary (id2word)
    from mstml._file_driver import read_pickle
    vocab_path = os.path.join(orch.dataset_directory, "clean", "id2word.pkl")
    id2word = read_pickle(vocab_path) if os.path.exists(vocab_path) else None

    n_show = min(6, len(cluster_to_topics))
    print(f"Showing {n_show} of {len(cluster_to_topics)} meta-topic clusters\n")

    for cid in sorted(cluster_to_topics)[:n_show]:
        topic_indices = cluster_to_topics[cid]
        # Average the topic-word distributions in this cluster
        cluster_phi = orch.topic_vectors[topic_indices].mean(axis=0)
        top_word_ids = cluster_phi.argsort()[::-1][:10]
        if id2word is not None:
            top_words = [id2word.get(i, str(i)) for i in top_word_ids]
        else:
            top_words = [f"word_{i}" for i in top_word_ids]
        print(f"Cluster {cid:2d} ({len(topic_indices):3d} topics): {', '.join(top_words)}")
else:
    print("topic_cluster_labels not available — run create_topic_embedding() first.")

## 14. Author Interdisciplinarity Scores

Rank authors by interdisciplinarity using entropy over diffused topic
distributions ψ_u.  Higher entropy → broader topic coverage → more
interdisciplinary.  This corresponds to §IV in the paper.

In [ ]:
# Compute entropy of each author's diffused distribution
from mstml._math_driver import entropy

author_entropy = {
    auth_id: entropy(dist)
    for auth_id, dist in orch.author_ct_distns.items()
}

sorted_authors = sorted(author_entropy.items(), key=lambda x: x[1], reverse=True)

print("Top-20 most interdisciplinary authors (by topic entropy):")
print(f"{'Rank':>4}  {'Author ID':>12}  {'Entropy':>8}")
print("-" * 32)
for rank, (auth_id, ent) in enumerate(sorted_authors[:20], 1):
    print(f"{rank:>4}  {auth_id:>12}  {ent:>8.4f}")

In [ ]:
# Distribution of author entropy scores
fig, ax = plt.subplots(figsize=(9, 4))
entropies = list(author_entropy.values())
ax.hist(entropies, bins=60, color="steelblue", edgecolor="white", linewidth=0.4)
ax.set_xlabel("Topic entropy H(ψ_u)")
ax.set_ylabel("Number of authors")
ax.set_title("Distribution of Author Interdisciplinarity Scores")
plt.tight_layout()
plt.savefig(os.path.join(EXPERIMENT_DIR, "fig_author_entropy_hist.pdf"), dpi=300)
plt.show()

## 15. Save All Results

In [ ]:
orch.save_topic_embedding()
results_path = orch.finalize_experiment()
print(f"All results saved to: {results_path}")

---

## Notes on Reproducing Paper Figures

The exact figures in the CAMSAP2025 paper were generated by the code in the
`AToMS-LP` repository.  This notebook reproduces the same pipeline using the
refactored MSTML library.  Minor visual differences may arise due to:

- **Random seeds** in LDA training (`random_state=42` in `config.yaml`)
- **PHATE stochasticity** — PHATE uses a random seed internally
- **arXiv snapshot date** — the paper used data available up to a specific
  crawl date; a newer snapshot will contain additional papers

To reproduce figures as closely as possible:
1. Use the exact same arXiv JSONL snapshot as the paper
2. Keep `random_state=42` in `config.yaml`
3. Use the same PHATE `random_state` (set via `topic_embedding.params.phate.random_state`)

If you need the exact AToMS-LP notebook that generated the published figures,
see `AToMS-LP/main/notebooks/`.